# 🦺 Detector de EPIs — Treinamento e Teste com YOLOv8

Este projeto permite trabalhar de **duas formas**. Leia esta orientação antes de executar as células.

## Escolha como deseja usar o projeto

### 🅰️ Modo A — Ainda não tenho um modelo treinado
Use as células originais **1 a 6** para instalar as bibliotecas, enviar o dataset `.zip`, conferir as classes, treinar o detector, testar fotos e baixar o modelo `.pt`.

### 🅱️ Modo B — Já tenho um modelo `.pt` salvo
Vá até a seção **“Modo B — Testar um modelo `.pt` já treinado”**, no final deste notebook. Execute a célula de instalação (Célula 1) se esta for uma sessão nova e, depois, execute as células do Modo B em ordem. Você enviará somente o arquivo `.pt` e as imagens de teste — **não precisa enviar o dataset nem treinar novamente**.

> **Importante:** os códigos das células originais foram mantidos sem alterações. As novas células foram acrescentadas ao final. Não execute as células de envio do dataset e treinamento se quiser apenas testar um modelo já pronto.

## 📦 Célula 1 — Instalar as bibliotecas e verificar o hardware

Execute esta célula primeiro em uma sessão nova do Google Colab, independentemente do modo escolhido.

In [ ]:
!pip install ultralytics matplotlib pyyaml -q

import torch
print('=' * 50)
print(f'✅ PyTorch versão: {torch.__version__}')
if torch.cuda.is_available():
    print(f'✅ Placa de vídeo detectada: {torch.cuda.get_device_name(0)}')
    print('🚀 Treinamento rodará em alta velocidade na GPU!')
else:
    print('⚠️ GPU não detectada! Se estiver no Google Colab, ative em:')
    print('   Ambiente de execução → Alterar tipo de ambiente de execução → T4 GPU')
print('=' * 50)


## 📂 Célula 2 — Enviar o dataset (Modo A)

**Use esta célula somente se escolheu o Modo A**, para treinar ou treinar novamente o modelo. Se já possui um arquivo `.pt` e quer apenas testar imagens, pule esta célula e vá para o Modo B no final do notebook.

In [ ]:
from google.colab import files
import zipfile
import shutil
from pathlib import Path

print('👇 Clique no botão abaixo e selecione o arquivo .zip do dataset:')
uploaded = files.upload()

if not uploaded:
    raise Exception('❌ Nenhum arquivo foi selecionado! Por favor, selecione o arquivo .zip.')

nome_zip = list(uploaded.keys())[0]
print(f'\n📦 Arquivo recebido: {nome_zip}')
print('📂 Descompactando e organizando os dados...')

pasta_destino = Path('/content/dataset_epis')
if pasta_destino.exists():
    shutil.rmtree(pasta_destino)
pasta_destino.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(nome_zip, 'r') as z:
    z.extractall(pasta_destino)

# Localiza automaticamente o data.yaml descompactado
yamls_encontrados = list(pasta_destino.rglob('data.yaml'))
if not yamls_encontrados:
    yamls_encontrados = list(Path('/content').rglob('data.yaml'))

if yamls_encontrados:
    DATA_YAML = str(yamls_encontrados[0])
    print(f'\n✅ Dataset pronto para treino! Arquivo de configuração: {DATA_YAML}')
else:
    raise FileNotFoundError('❌ Não foi encontrado nenhum arquivo data.yaml dentro do zip enviado.')


## 🔍 Célula 3 — Conferir as classes e os dados do dataset (Modo A)

Esta etapa ajuda a conferir as classes encontradas no dataset antes do treinamento. Execute-a somente no Modo A, depois de enviar e extrair o `.zip`.

In [ ]:
import yaml
from pathlib import Path
from collections import Counter

with open(DATA_YAML, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

classes = cfg.get('names', [])
if isinstance(classes, dict):
    classes = [classes[k] for k in sorted(classes.keys())]

print('=' * 60)
print(f'📋 TOTAL DE CLASSES DO DATASET: {len(classes)}')
print('=' * 60)
for i, nome in enumerate(classes):
    print(f'   [{i:2d}] {nome}')

# Contagem de amostras nas anotações de treino
labels_treino = list(Path(DATA_YAML).parent.rglob('train/labels/*.txt')) + list(Path(DATA_YAML).parent.rglob('labels/train/*.txt'))
if labels_treino:
    contador = Counter()
    for lf in labels_treino:
        with open(lf, 'r', encoding='utf-8') as f:
            for linha in f:
                partes = linha.strip().split()
                if partes:
                    contador[int(partes[0])] += 1
    
    print('\n📊 DISTRIBUIÇÃO DAS CLASSES NO TREINO:')
    for i, nome in enumerate(classes):
        qtd = contador[i]
        status = '✅ Boa quantidade' if qtd >= 40 else '⚠️ Poucas amostras'
        print(f'   {nome:30s}: {qtd:4d} caixas ({status})')


## 🚀 Célula 4 — Treinar o detector (Modo A)

Esta célula inicia um novo treinamento usando o dataset enviado. O processo pode demorar. **Não execute esta célula no Modo B**, pois nesse modo você já possui o modelo treinado.

In [ ]:
from ultralytics import YOLO
import torch
import time

print('=' * 65)
print('🚀 INICIANDO TREINAMENTO YOLOv8 PARA DETECÇÃO DE EPIs')
print('=' * 65)

# Detecta automaticamente a máquina (GPU CUDA se houver, caso contrário CPU)
device = 0 if torch.cuda.is_available() else 'cpu'
print(f'⚙️ Dispositivo de processamento: {device}')

model = YOLO('yolov8s.pt')
inicio = time.time()

results = model.train(
    data=DATA_YAML,
    epochs=50,          # 50 épocas para boa convergência
    imgsz=640,          # resolução padrão de treino
    batch=16,
    device=device,
    project='EPI_Detector_Projeto',
    name='treino_epis_completo',
    exist_ok=True,
    save=True,
    plots=True,
    seed=42,
    verbose=True
)

duracao = (time.time() - inicio) / 60
print(f'\n✅ Treinamento finalizado com sucesso em {duracao:.1f} minutos!')


## 🖼️ Célula 5 — Testar fotos (fluxo original do Modo A)

No Modo A, use esta célula depois do treinamento para testar fotos; ela também permite importar um `.pt` se não houver pesos de treinamento na sessão.

Se quiser seguir o fluxo separado e mais simples de importar um modelo pronto e depois enviar as fotos, use as células novas do **Modo B**, no final do notebook.

In [ ]:
from google.colab import files
from ultralytics import YOLO
import matplotlib.pyplot as plt
from pathlib import Path
import torch

# =====================================================================
# 1. CARREGAR O MODELO (AUTOMÁTICO OU VIA UPLOAD)
# =====================================================================
pesos_locais = list(Path('/content').rglob('*best.pt')) + list(Path('.').rglob('*best.pt'))

if pesos_locais:
    caminho_modelo = str(pesos_locais[0])
    print(f'✅ Usando modelo gerado no treinamento: {caminho_modelo}')
else:
    print('👇 Não foi encontrado treino ativo. Envie seu arquivo de modelo (.pt) do computador:')
    upload_modelo = files.upload()
    if not upload_modelo:
        raise Exception('❌ Nenhum modelo foi selecionado!')
    caminho_modelo = list(upload_modelo.keys())[0]
    print(f'✅ Modelo importado do computador: {caminho_modelo}')

# Dispositivo automático (GPU ou CPU sem travar em nenhuma máquina)
device = 0 if torch.cuda.is_available() else 'cpu'
modelo = YOLO(caminho_modelo)
print(f'⚙️ Dispositivo de inferência: {device}')

# =====================================================================
# 2. IMPORTAR AS FOTOS DOS FUNCIONÁRIOS PARA TESTE
# =====================================================================
print('\n👇 Clique no botão abaixo para escolher as FOTOS DOS FUNCIONÁRIOS:')
upload_fotos = files.upload()

if not upload_fotos:
    print('⚠️ Nenhuma foto foi enviada para teste.')
else:
    # CALIBRAÇÃO PARA CORPO INTEIRO:
    # conf=0.15: captura botas e luvas sem descartar detecções distantes
    # imgsz=1024: garante que objetos pequenos tenham nitidez máxima
    CONF_THRESHOLD = 0.15
    IMG_SIZE = 1024
    
    for nome_foto in upload_fotos.keys():
        print(f'\n🔍 Analisando: {nome_foto}')
        res = modelo.predict(
            source=nome_foto,
            conf=CONF_THRESHOLD,
            imgsz=IMG_SIZE,
            device=device,
            verbose=False
        )[0]
        
        plt.figure(figsize=(10, 10))
        plt.imshow(res.plot()[..., ::-1])
        plt.axis('off')
        plt.title(f'Detecção de EPIs: {nome_foto}', fontsize=12, fontweight='bold')
        plt.show()
        
        print(f'📋 EPIs detectados em {nome_foto}:')
        if len(res.boxes) == 0:
            print('   ⚠️ Nenhum EPI detectado com o limiar atual.')
        else:
            for box in res.boxes:
                cls_name = modelo.names[int(box.cls)]
                conf = float(box.conf)
                print(f'   ✅ {cls_name}: {conf:.1%}')
        print('-' * 55)


## 💾 Célula 6 — Baixar o modelo treinado (Modo A)

Use esta célula depois do treinamento para baixar os pesos `.pt` para seu computador. Depois, você poderá reutilizá-los no Modo B sem treinar novamente.

In [ ]:
from google.colab import files
import shutil
from pathlib import Path

encontrados = list(Path('/content').rglob('*best.pt')) + list(Path('.').rglob('*best.pt'))
if encontrados:
    melhor_modelo = encontrados[0]
    destino_download = '/content/best_epi_yolov8.pt'
    shutil.copy(str(melhor_modelo), destino_download)
    print(f'⬇️ Baixando seu modelo treinado ({melhor_modelo})...')
    files.download(destino_download)
else:
    print('❌ Arquivo de modelo (.pt) não foi localizado. Execute o treinamento primeiro.')


---

# 🅱️ Modo B — Testar um modelo `.pt` já treinado

Use esta seção quando você já tem um arquivo de modelo, por exemplo `best_epi_yolov8.pt`, salvo no computador.

**Ordem de execução:**
1. Se esta for uma sessão nova, execute a **Célula 1** no início do notebook para instalar as bibliotecas.
2. Execute a nova **Célula B1** para importar o arquivo `.pt`.
3. Execute a nova **Célula B2** para escolher uma ou mais imagens e visualizar as detecções.
4. Se quiser salvar os resultados no computador, execute a nova **Célula B3**.

Neste modo, não execute as células originais de envio do dataset, diagnóstico ou treinamento. O dataset não é necessário para fazer previsões com um modelo já treinado.

## 🧩 Célula B1 — Importar e carregar o modelo `.pt`

Clique em **Escolher arquivos** e selecione o modelo treinado no seu computador. Esta etapa apenas carrega os pesos; não inicia treinamento.

In [ ]:
from google.colab import files
from ultralytics import YOLO
from pathlib import Path
import torch

print('👇 Selecione seu modelo treinado (.pt), por exemplo best_epi_yolov8.pt:')
upload_modelo_pronto = files.upload()

if not upload_modelo_pronto:
    raise Exception('Nenhum modelo foi selecionado. Execute esta célula novamente e escolha um arquivo .pt.')

caminho_modelo_pronto = next(iter(upload_modelo_pronto.keys()))
if not caminho_modelo_pronto.lower().endswith('.pt'):
    raise ValueError('O arquivo selecionado não parece ser um modelo .pt. Selecione o arquivo de pesos treinados.')

modelo_pronto = YOLO(caminho_modelo_pronto)
dispositivo_teste = 0 if torch.cuda.is_available() else 'cpu'

print(f'✅ Modelo carregado: {caminho_modelo_pronto}')
print(f'⚙️ Dispositivo para detecção: {dispositivo_teste}')
print('📋 Classes conhecidas pelo modelo:')
print(modelo_pronto.names)

## 🖼️ Célula B2 — Enviar fotos e testar o modelo

Você pode selecionar uma ou várias imagens. Envie somente as fotos que deseja analisar, não o dataset inteiro.

In [ ]:
from google.colab import files
from pathlib import Path
import matplotlib.pyplot as plt

print('👇 Selecione uma ou mais imagens para testar:')
upload_imagens_prontas = files.upload()

if not upload_imagens_prontas:
    raise Exception('Nenhuma imagem foi selecionada. Execute esta célula novamente e escolha uma ou mais fotos.')

# Ajustes da detecção: diminua conf se quiser aceitar detecções menos confiantes.
CONFIANCA_MINIMA = 0.25
TAMANHO_IMAGEM = 640

resultados_modo_b = []
for nome_imagem in upload_imagens_prontas.keys():
    print(f'\n🔎 Analisando: {nome_imagem}')
    resultado = modelo_pronto.predict(
        source=nome_imagem,
        conf=CONFIANCA_MINIMA,
        imgsz=TAMANHO_IMAGEM,
        device=dispositivo_teste,
        verbose=False
    )[0]
    resultados_modo_b.append((nome_imagem, resultado))

    imagem_anotada = resultado.plot()
    plt.figure(figsize=(12, 8))
    plt.imshow(imagem_anotada[:, :, ::-1])
    plt.axis('off')
    plt.title(f'Detecção de EPIs: {nome_imagem}')
    plt.show()

    print('📋 Objetos detectados:')
    if len(resultado.boxes) == 0:
        print('   Nenhuma detecção acima do nível de confiança definido.')
    else:
        for caixa in resultado.boxes:
            classe = modelo_pronto.names[int(caixa.cls[0])]
            confianca = float(caixa.conf[0])
            print(f'   • {classe}: {confianca:.1%} de confiança')
    print('-' * 50)

print('✅ Teste concluído. Para salvar as imagens com as caixas, execute a Célula B3.')

## 💾 Célula B3 — (Opcional) Baixar as imagens com as detecções

Esta célula cria arquivos de imagem com as caixas desenhadas e baixa um arquivo `.zip` com todos os resultados.

In [ ]:
from pathlib import Path
from PIL import Image
import shutil

pasta_saida_modo_b = Path('/content/resultados_teste_epi')
pasta_saida_modo_b.mkdir(parents=True, exist_ok=True)

for nome_imagem, resultado in resultados_modo_b:
    imagem_bgr = resultado.plot()
    imagem_rgb = imagem_bgr[:, :, ::-1]
    nome_saida = Path(nome_imagem).stem + '_deteccoes.jpg'
    caminho_saida = pasta_saida_modo_b / nome_saida
    Image.fromarray(imagem_rgb).save(caminho_saida, quality=95)
    print(f'✅ Salvo: {caminho_saida}')

arquivo_zip_resultados = shutil.make_archive('/content/resultados_teste_epi', 'zip', pasta_saida_modo_b)
files.download(arquivo_zip_resultados)

## 🎤 Resumo para explicar o projeto em sala

O projeto tem dois fluxos. No primeiro, enviamos o dataset anotado, verificamos suas classes e treinamos o YOLOv8 para reconhecer EPIs. Ao terminar, podemos baixar os pesos do modelo em um arquivo `.pt`. No segundo fluxo, reutilizamos esse arquivo já treinado: carregamos o `.pt`, enviamos fotos novas e o modelo mostra as detecções, com caixas, nomes das classes e níveis de confiança. Assim, para testar novas imagens, não é necessário enviar novamente o dataset nem repetir o treinamento.

**Observação:** a qualidade das detecções depende das classes e dos exemplos usados para treinar o modelo. O arquivo `.pt` contém os pesos aprendidos, não o dataset completo.